# Laser pulse-shape analysis: OMEGA shot 93305

The OMEGA laser fires 60 beams at a target at the same time. A streak camera called P510 records
how the power of each beam changes in time: the **pulse shape**.
In this session you start from the camera's **raw images** and work step by step to the power curve
of every beam, then average them and compute statistics.

**Part 1** takes 20 beams through the whole analysis, from raw images to statistics.
**Part 2** applies the last steps to all 60 beams of the shot.

**How to use this notebook.** Click a grey code cell and press `Shift + Enter` to run it.
Run the cells from top to bottom. Every step draws a figure.

**If you change a parameter**, run that cell **and every cell below it** again.
Otherwise the later steps still use the old result.

| Step | What you do | What you get |
|---|---|---|
| 1 | Load the data | Raw CCD images |
| 2 | Extract the curves | One curve per beam: x = pixel, y = counts |
| 3 | Calibrate time | x axis in picoseconds (ps) |
| 4 | Calibrate power | y axis in terawatts (TW) |
| 5 | Align in time | All curves share one time origin |
| 6 | Average | The mean power curve |
| 7 | Statistics | Spread of energy, peak power and power between beams |

In [ ]:
import numpy as np
import pulse   # the helper functions for this session live in pulse.py

# Part 1: from raw images to statistics (20 beams)

## Step 1: load the data

Each P510 camera takes one image. In the image, **time runs from left to right** and **space runs from top to bottom**.
The 10 bright bands in the middle are 10 laser beams.
The row of small spots along the top and bottom edges is a **timing comb (fiducial)**: neighbouring spots are 548.7 ps apart.

We have images from two cameras (cluster 1 and cluster 2), so 20 beams in total: beams 10 to 19 and 20 to 29.
Each file holds two frames, the signal and the background. The code subtracts the background for you.

The images use a **log colour scale**: the combs are about 10 times fainter than the beams and would be almost invisible on a linear scale.

In [ ]:
shots = pulse.load_raw('data/P510_*-s93305.h5')

## Step 2: extract the raw curves

### 2a. Straighten the images

The camera is not mounted perfectly straight, so the beam bands are slightly tilted.
If a band is tilted, adding up its rows mixes the edge of one beam into its neighbour.
Rotate each image so the dark lines between beams run horizontally.

The zoomed figure stretches the rows so a tilt of 1 degree is easy to see. The red dashed lines are horizontal.
The printout measures the tilt of the dark lines before and after rotation.

**Exercise.** Set both angles to `0.0` and look at cluster 2. Then try a few values and get the tilt after rotation close to 0.
In cluster 1 the lines fan out (the top ones tilt one way, the bottom ones the other way). Can a rotation fix that?

We call the result `rotated`, not `shots`, so running this cell twice does not rotate the images twice.

In [ ]:
rotated = pulse.rotate(shots, angles={1: 0.2, 2: -1.7})   # degrees, one angle per cluster

### 2b. Add up the rows of each beam

For each bright band, **add up the pixels in every column**. This gives one curve per beam.
Both axes are still "raw": the x axis is the pixel number and the y axis is CCD counts.

The figure has one row per camera. Left: the raw image, with a coloured box around the rows that are added up for each beam,
labelled with the beam number and the width of the box (about 38 rows). We leave out a few rows at the edges of each band,
where the dark gap between two beams starts. Right: the curves of that camera, in the same colours.

In [ ]:
raw = pulse.extract_lineouts(rotated)

Twenty curves on one plot are hard to read. Choose the beams you want to see with `beams=`:

- `beams='all'`: every beam
- `beams=12`: one beam
- `beams=[10, 19, 25]`: a list of beams
- `beams=range(20, 30)`: beams 20 to 29 (the last number is not included)

The beams we have are 10 to 29. The image shows only the cameras of the chosen beams, with only their rows boxed.
For the result of every later step, use `.plot(beams=...)` instead, for example `power.plot(beams=12)`.

**Think about it.** Why are some curves taller than others? Is that a difference between the beams, or between parts of the camera?

In [ ]:
pulse.show_lineouts(raw, rotated, beams=[10, 15, 20, 25])

## Step 3: calibrate the time axis (pixel → ps)

We know the pulses of the timing comb are 548.7 ps apart.
If we find which pixel each comb pulse sits on, we can fit the relation `t(pixel)`. We do it in three small steps.

### 3a. Find the comb pulses

Average the rows of the top comb and of the bottom comb (boxed on the image, left). Each gives a line of 8 peaks (right).
The triangles mark the pixel of each peak (its centroid).

In [ ]:
combs = pulse.find_combs(rotated)

### 3b. Fit time against pixel

Pulse number $k$ arrives at $t_k = k \times 548.7$ ps. Fit a polynomial $t(\text{pixel})$ through the points ($\text{pixel}_k$, $t_k$).

- Left: the comb pulses (dots) and the fitted curve.
- Middle: the slope of the fit, i.e. how many ps one pixel represents. It is **not constant**, because the camera's sweep speed is not uniform.
- Right: the fit residuals, i.e. how far each comb pulse is from the fitted curve. This is the size of our timing error.

**Exercise.** Change `degree=2` to `1` (a straight line) or `3`, and run the cells again. How do the residuals change?

In [ ]:
fits = pulse.fit_time(combs, period_ps=548.7, degree=2)

### 3c. Give each beam its own time axis

The top and bottom combs give slightly different sweep speeds, so each beam uses a mix of the two fits,
weighted by its position: $t = (1-w)\,t_\text{top} + w\,t_\text{bottom}$, where $w$ goes from 0 at the top comb to 1 at the bottom comb.

In [ ]:
timed = pulse.apply_time(raw, rotated, fits)

## Step 4: calibrate the power axis (counts → TW)

Calorimeters measured the total UV energy $E$ of every beam (in joules).
Integrating the power curve over time must give that energy, so

$$P(t) = E \cdot \frac{S(t)}{\int S\,dt}$$

where $S(t)$ is the raw curve from step 2. With $E$ in J and $t$ in ps, $P$ comes out in TW (1 J/ps = 1 TW).
We do it in three small steps.

### 4a. The measured energy $E$ of each beam

In [ ]:
energy = pulse.load_beam_energy('data/P510-p510_data_93305.h5')   # {beam number: energy in J}, all 60 beams
E = np.array([energy[b] for b in timed.beams])                     # the energies of our 20 beams, in the same order
pulse.per_beam(timed.beams, E, 'energy E (J)')

### 4b. The area $\int S\,dt$ under each raw curve

Negative counts are only background noise, so we set them to 0 first.
The ratio $E / \int S\,dt$ is the number of TW that one count stands for. If all channels of the camera were equally sensitive,
it would be about the same for every beam. Look at the spread: this answers the question in step 2.

In [ ]:
S = timed.P.clip(min=0)               # raw curves, negative counts set to 0
area = pulse.integrate(timed.t, S)    # ∫ S dt for each beam (counts × ps)
scale = E / area                      # TW per count
pulse.per_beam(timed.beams, scale, 'E / ∫S dt (TW per count)')

### 4c. Scale each curve

In [ ]:
power = pulse.Curves(timed.t, S * scale[:, None], timed.beams, 'ps', 'TW')   # P(t) = S(t) × E / ∫S dt
power.plot('Step 4: power axis calibrated')

## Step 5: align in time

The beams arrive at slightly different times (tens of ps), and the two cameras have different time origins.
We pick one feature of every curve, shift the curve so that this feature sits at t = 0,
then put all curves on the same time grid. There is more than one sensible feature:

| `edge` | t = 0 is where the curve... |
|---|---|
| `'rise'` | first rises through `level` × its peak (default: 5%) |
| `'fall'` | last falls through `level` × its peak, e.g. `level=0.5` for the 50% point of the falling edge |
| `'peak'` | reaches its maximum (`level` is not used) |

**Exercise.** Try `edge='rise', level=0.2`, then `edge='fall', level=0.5`, then `edge='peak'`.
Run steps 6 and 7 again each time. How do the mean pulse and the power balance change? Which choice makes the beams agree best, and why?

In [ ]:
aligned = pulse.align(power, edge='rise', level=0.05)

## Step 6: average

At each moment in time, average the power over all beams. The shaded band is ±1 standard deviation (STDEV).

In [ ]:
mean_pulse = pulse.average(aligned)

## Step 7: statistics

**First figure: one dot per beam**, for its energy (top) and its peak power (bottom).
The black line is the mean of all beams, and the shaded bands are the mean ±1%, ±5% and ±10%.
The printout counts how many beams fall inside each band, and gives **STDEV/mean**, the relative spread.

**Second figure: the pulse.** Top: every beam (grey), the mean (black) and the bands mean ±1% and ±5%.
Bottom: STDEV/mean at each moment, over the main pulse (where the mean is above 10% of its peak).
Its average over time is the **power balance**.
It tells us whether all beams push on the target equally, which matters for a symmetric implosion.

**Think about it.** Where does the energy STDEV come from? (Hint: what did we use to calibrate power in step 4?)

In [ ]:
stats = pulse.statistics(aligned)

# Part 2: the whole shot, 60 beams

Part 1 showed the full analysis on the 20 beams we have raw images for.
OMEGA's own software has processed **all 60 beams** with the same kind of calibration.
We now put that result through **the same steps 5 to 7**.

In [ ]:
official = pulse.load_official('data/P510-p510_data_93305.h5')
official_aligned = pulse.align(official, edge='rise', level=0.05)
official_mean = pulse.average(official_aligned)

### Statistics of all 60 beams

With 60 beams the statistics become more reliable.
A rough rule: the relative uncertainty of a STDEV estimated from $n$ values is about $1/\sqrt{2(n-1)}$,
which is about 16% for 20 beams and 9% for 60 beams.

In [ ]:
official_stats = pulse.statistics(official_aligned)

**Think about it.**
Step 5 removed the differences in arrival time between beams, so the power balance above only reflects differences in **pulse shape**.
Without alignment, on the common time axis, the power balance of the 60 beams is about 8.9%, not about 6.7%.
Which number matters for the target? **Your analysis choices change the quantity you measure.**